In [190]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [191]:
df = pd.read_csv('./content/data/train.csv')

In [192]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [193]:
print(f'Count features with null-value = {df.isnull().any(axis=0).sum()}')

Count features with null-value = 19


Очень много фичей, много интересных мыслей появлялось в процессе анализа. Щас всё попробую. Но сначала надо избавиться от null-значений.

# Работа с null-значений
В начале анализа выяснил, что много фичей имеют много null-значений. Сначала казалось это плохо, но потом узнал, что эти null-значения показывают отсутствие данного
объекта на участке (напр, нет забора, камина и т.п.). Это не значит что данные были потеряны.\
Поэтому важно все эти null-значения заменить на что-то логично, с чем модели могут работать.

In [194]:
null_count = df.isnull().sum()
null_count[null_count > 0]

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64

## LotFrontage null
Показывает ширину участка, граничащую с улицей.\
Очевидно, дом не может не граничить с улицей, т.к. какой-то подход должен к нему быть. Поэтому стоит эти значения заполнить.

Можно было бы заполнить все null-значения обычным средним, но хочется большей точности. У каждого сэмпла есть *LotConfig*, которые показывает, насколько
хорошо дом прилегает к дороге. Думаю, у домов, которые прилегают к дороге с 3-х сторон, значение *LotFrontage* будет заметно выше:

In [195]:
df.groupby('LotConfig')['LotFrontage'].mean()

LotConfig
Corner     84.039801
CulDSac    59.911111
FR2        63.515152
FR3        70.750000
Inside     67.715686
Name: LotFrontage, dtype: float64

Интересно, что угловой участок (*Corner*) имеет большее значение, чем участок, окружённый улицами с 3-х сторон (*FR3*). Но оба они всё равно имеют большее значение,
чем другие.

In [196]:
lotfrontage_null = df.groupby('LotConfig')['LotFrontage'].mean().round()

for value, mean in lotfrontage_null.items():
    df.loc[(df['LotFrontage'].isnull()) & (df['LotConfig'] == value), 'LotFrontage'] = mean

## Alley null
Показывает либо материал аллеи, либо её отсутствие. Здесь null-значение несёт важную информацию.

In [197]:
df.loc[df['Alley'].isnull(), 'Alley'] = 'Absent'

## MasVnrType null
Показывает облицовку дома. Null-значение несёт в себе информацию.

In [198]:
df.loc[df['MasVnrType'].isnull(), 'MasVnrType'] = 'Absent'

## MasVnrArea null
Показывает площадь облицовки из кирпица. Имеет 8 null-значений, хотя отсутствие облицовки должно просто обозначатьс цифрой 0.

In [199]:
df[df['MasVnrArea'].isnull()][['MasVnrType', 'MasVnrArea']]

,MasVnrType,MasVnrArea
234,Absent,NaN
529,Absent,NaN
650,Absent,NaN
936,Absent,NaN
973,Absent,NaN
977,Absent,NaN
1243,Absent,NaN
1278,Absent,NaN


У null-значений облицовка отсутствует, так что можно спокойно присваивать значение 0:

In [200]:
df.loc[df['MasVnrArea'].isnull(), 'MasVnrArea'] = 0

## BsmtQual null
Показывает высоту подвала. Null-значения показывают отсутствие подвала.

In [201]:
df.loc[df['BsmtQual'].isnull(), 'BsmtQual'] = 'Absent'

## BsmtCond null
Оценивает общее состояние подвала. Null-значение показывает отсутствие подвала.

In [202]:
df.loc[df['BsmtCond'].isnull(), 'BsmtCond'] = 'Absent'

## BsmtExposure null
Показывает стену, ведущую на террасу или в сад. Т.е. насколько хорошо подвал выходит на наружу: его может быть совсем не видно, могут быть небольшие окна, из которых попадает свет,
а может быть целая дверь с выходом во двор.

У 37 домов подвалы отсутсвуют, точно будет 37 null-значений. Но есть и ещё один:

In [203]:
df[(df['BsmtExposure'].isnull()) & (df['BsmtQual'] != 'Absent')].filter(regex=r'.*Bsmt.*')

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,BsmtFullBath,BsmtHalfBath
948,Gd,TA,NaN,Unf,0,Unf,0,936,936,0,0


Видно, что подвал есть и достаточно большой. Но значения *BsmtFin\** показывают, что подвал недостроен. Поэтому фактически он есть, но невозможно использовать.\
Его тоже можно пометить как *Absent*:

In [204]:
df.loc[df['BsmtExposure'].isnull(), 'BsmtExposure'] = 'Absent'

## BsmtFinType1 null
Качество отделки основной зоны подвала. 37 null-значений. Те самые дома, у которых подвал отсутствует.

In [205]:
df.loc[df['BsmtFinType1'].isnull(), 'BsmtFinType1'] = 'Absent'

## BsmtFinType2 null
Качество отделки дополнительной зоны подвала. 38 null-значений. У 37 домов точно нет подвала. 38-й сэмпл очень интересный:

In [206]:
df[(df['BsmtFinType2'].isnull()) & (df['BsmtQual'] != 'Absent')].filter(regex=r'.*Bsmt.*')

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,BsmtFullBath,BsmtHalfBath
332,Gd,TA,No,GLQ,1124,NaN,479,1603,3206,1,0


Это какое-то потерянное значение. У данного дома есть подвал, от отделан, достаточно хорошая площадь. Даже видно, что дополнительная зона подвала готова
к использованию (судя по *BsmtFinSF2*), но при этом значение *BsmtFinType2* отсутствует. Заполним его самым популярным значением в *BsmtFinType2*:

In [207]:
df['BsmtFinType2'].value_counts()

BsmtFinType2
Unf    1256
Rec      54
LwQ      46
BLQ      33
ALQ      19
GLQ      14
Name: count, dtype: int64

Самое популярное значение *Unf*, но мы видим, что дополнительная зона подвала точно готова к использованию. Использую лучше значение *Rec*:

In [208]:
df.loc[(df['BsmtFinType2'].isnull()) & (df['BsmtFinSF2'] > 0), 'BsmtFinType2'] = 'Rec'

In [209]:
df.loc[df['BsmtFinType2'].isnull(), 'BsmtFinType2'] = 'Absent'

## Electrical null
Показывает тип электрической системы.

In [210]:
df[df['Electrical'].isnull()]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
1379,1380,80,RL,73.0,9735,Pave,Absent,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2008,WD,Normal,167500


Ничего необычного, заполню самым частым значением:

In [211]:
df['Electrical'].value_counts()

Electrical
SBrkr    1334
FuseA      94
FuseF      27
FuseP       3
Mix         1
Name: count, dtype: int64

In [212]:
df.loc[df['Electrical'].isnull(), 'Electrical'] = 'SBrkr'

## FireplaceQu null
Показывает качество каминов. Null-значение показывает отсутствие камина в доме.

Проверим сэмплы, у которых качество камина имеет null-значение, а количество каминов > 0:

In [213]:
df[(df['FireplaceQu'].isnull()) & (df['Fireplaces'] > 0)]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice


В датасете отсутствуют подобные сэмплы, можно спокойно обозначать *Absent*:

In [214]:
df.loc[df['FireplaceQu'].isnull(), 'FireplaceQu'] = 'Absent'

## Garage* null
За гараж отвечают признаки *GarageType*, *GarageYrBlt*, *GarageFinist*, *GarageQual* и *GarageCond*.\
Каждый из них имеет 81 null-значений. Они показывают отсутствие гаража. Совпадение null-значений в каждом признаки показывает отсутствие пропусков.

In [215]:
df.loc[df['GarageType'].isnull() , 'GarageType'] = 'Absent'
df.loc[df['GarageFinish'].isnull() , 'GarageFinish'] = 'Absent'
df.loc[df['GarageQual'].isnull() , 'GarageQual'] = 'Absent'
df.loc[df['GarageCond'].isnull() , 'GarageCond'] = 'Absent'

И с фичой *GarageYrBlt* получается неоднозначная ситуация: столбец содержит числа (float64), однако при отсутствии гаража стоит null-значение. Изменить на *Absent* не получится.\
Если подобрать какое-то определённое число для null-значений (напр, 0 или год постройки дома), разные модели могут неправильно обучаться на них.

Думаю, хорошим вариантом будет изменить *GarageYrBlt* на возраст гаража *GarageAge*, а для наличия гаража использовать новую фичу *HasGarage*:

In [216]:
df['GarageAge'] = (df['YrSold'] - df['GarageYrBlt'])
df['GarageAge'] = df['GarageAge'].fillna(0)
df['GarageAge'] = df['GarageAge'].astype(int)

df['HasGarage'] = (df['GarageArea'] > 0).astype(int)

df = df.drop(columns='GarageYrBlt')

## PoolQC null
Показывает качество бассейна. Null-значение показывает отсутствие бассейна.

In [217]:
df.loc[df['PoolQC'].isnull(), 'PoolQC'] = 'Absent'

## Fence null
Показывает качество забора. Null-значение показывает отсутствие забора.

In [218]:
df.loc[df['Fence'].isnull(), 'Fence'] = 'Absent'

## MiscFeature null
Показывает наличие дополнительного функционала, которые не вошёл в основные фичи. Есть ещё фича *MiscValue*, которая показывает стоимость этого функционала.

Думаю, нет смысла указывать что за объект стоит на участке, если его ценность уже указана в *MiscValue*. А объкты, у которых нет ничего дополнительного, имеют
*MiscValue* = 0, что и показывает отсутствие каких-либо доп.объектов на участке.\
Поэтому удалю столбец *MiscFeature*:

In [219]:
df = df.drop(columns='MiscFeature')

# Преобразование фичей
_UPD 1_: Я тут щас преобразовывал фичи из категориальных в ординальные и понял, что не всем моделям подойдёт ординальное кодирование. Линейным моделям (а следовательно, и нейросетям)
лучше делать _one-hot-encoding_. Благо для этого есть _.get_dummies()_.\
Поэтому я продолжу здесь делать заготовки для _ordinal-encoding_, который пойдёт потом в пайплайн, но итоговое преобразование будет зависить от модели.

*UPD 2*: При реализации *Ordinal-encoding* значения отсортированы с помощью вещественные чисел. Т.е. значимость признаков идёт не \[$1, 2, 3, \dots$], а \[$1, 2.3, 4.5, \dots$].
Это было сделано чтобы не просто расположить их в порядке возрастания ценности, но и чтобы обозначить моделям, насколько выростает значимость у каждого значения.

 ## MSSubClass fe
Признак содержит числа, обозначающие какой-то тип дома. Расположены в следующем порядке:
- 20  — 1-STORY 1946 & NEWER ALL STYLES
- 30  — 1-STORY 1945 & OLDER
- 40  — 1-STORY W/FINISHED ATTIC ALL AGES
- 45  — 1-1/2 STORY - UNFINISHED ALL AGES
- 50  — 1-1/2 STORY FINISHED ALL AGES
- 60  — 2-STORY 1946 & NEWER
- 70  — 2-STORY 1945 & OLDER
- 75  — 2-1/2 STORY ALL AGES
- 80  — SPLIT OR MULTI-LEVEL
- 85  — SPLIT FOYER
- 90  — DUPLEX - ALL STYLES AND AGES
- 120 — 1-STORY PUD (Planned Unit Development) - 1946 & NEWER
- 150 — 1-1/2 STORY PUD - ALL AGES
- 160 — 2-STORY PUD - 1946 & NEWER
- 180 — PUD - MULTILEVEL - INCL SPLIT LEV/FOYER
- 190 — 2 FAMILY CONVERSION - ALL STYLES AND AGES

Но возрастание числа не соответствует росту ценности дома.\
Можно было бы щтательно исследовать данную тему в интернете и поставить дома в возрастании в соответствии с данными. Но будет проще найти среднюю стоимость каждого дома и
расположить их в порядке возрастания средней стоимости дома:

In [220]:
df.groupby('MSSubClass', as_index=False)['SalePrice'].mean().sort_values('SalePrice')

,MSSubClass,SalePrice
1,30,95829.724638
13,180,102300.000000
3,45,108591.666667
14,190,129613.333333
10,90,133541.076923
12,160,138647.380952
4,50,143302.972222
9,85,147810.000000
2,40,156125.000000
6,70,166772.416667


In [221]:
df['MSSubClass_Rating'] = 0

mssubclass_sorted = df.groupby('MSSubClass', as_index=False)['SalePrice'].mean().sort_values('SalePrice')

for idx, mssubclass in enumerate(mssubclass_sorted['MSSubClass']):
    df.loc[df['MSSubClass'] == mssubclass, 'MSSubClass_Rating'] = idx + 1

Проверим правильность сортировки:

In [222]:
table_old = df.groupby('MSSubClass', as_index=False)['SalePrice'].mean().sort_values('SalePrice').reset_index(drop=True)
table_new = df.groupby('MSSubClass_Rating', as_index=False)['SalePrice'].mean().sort_values('SalePrice').reset_index(drop=True)
pd.concat([table_old, table_new], axis=1)

,MSSubClass,SalePrice,MSSubClass_Rating,SalePrice
0,30,95829.724638,1,95829.724638
1,180,102300.000000,2,102300.000000
2,45,108591.666667,3,108591.666667
3,190,129613.333333,4,129613.333333
4,90,133541.076923,5,133541.076923
5,160,138647.380952,6,138647.380952
6,50,143302.972222,7,143302.972222
7,85,147810.000000,8,147810.000000
8,40,156125.000000,9,156125.000000
9,70,166772.416667,10,166772.416667


In [223]:
df = df.drop(columns='MSSubClass')

## MSZoning fe
Показывает градостроительное назначение земли.

В данном признаки 8 значений, но в обучаемых данных используется лишь 5. Если модель увидит какое-то новое значение, предсказание
от этого лучше не станет. Но и выдавать ему значение другой группы тоже не справедливо. Буду присвать ему сренее значение, если кодировка будет с помощью *Ordinal-encoding*.
Порядок в данной кодировке (из EDA): *C (all)* < *RH* < *RM* < *RL* < *FV*.

А вообще, для данного признака лучше всего выглядит *One-hot-encoding*.

In [224]:
df['MSZoning_Rating'] = 2.5 # Сразу присваиваю значение по умолчанию

df.loc[df['MSZoning'] == 'C (all)', 'MSZoning_Rating'] = 1
df.loc[df['MSZoning'] == 'RH', 'MSZoning_Rating'] = 2
df.loc[df['MSZoning'] == 'RM', 'MSZoning_Rating'] = 3
df.loc[df['MSZoning'] == 'RL', 'MSZoning_Rating'] = 4
df.loc[df['MSZoning'] == 'FV', 'MSZoning_Rating'] = 5

df = df.drop(columns='MSZoning')

## Street fe
Показывает материал дороги улицы, из которого она сделана.

В датасете 1460 сэмплов, 6 из который имеют значение *Grvl*, остальные — *Pave*. Признак явлется квазиконстантным. Следует удалить:

In [225]:
df = df.drop(columns='Street')

## Alley fe
Показывает материал аллеи, из которого она сделана.

Посчитаем среднюю цену дома с разным типом алей:

In [226]:
df.groupby('Alley')['SalePrice'].mean()

Alley
Absent    183452.131483
Grvl      122219.080000
Pave      168000.585366
Name: SalePrice, dtype: float64

При отсутствии аллеи дом стоит в среднем 183к$. Дома с твёрдой аллеей стоят дороже чем с рассыпчатой. Но домов с аллей в принципе мало: всего 91 штука (примерно 7% от всего датасета).
Более интересен сам факт наличия данной аллеи: дома без аллеи стоят, 183к\\$, а с аллеей — 145к\\$. Разница ощутима.\
Поменяю фичу *Alley* на *Has_Alley*, которая будет показывать факт наличия аллеи у дома:

In [227]:
df['Has_Alley'] = (df['Alley'] != 'Absent').astype(int)

df = df.drop(columns='Alley')

## LotShape fe
Общая форма объекта недвижимости. Показывает рост цен от *Reg* до *IR2*, а на *IR3* резкое падение.

*Ordinal-encoding* не подойдёт. В идеале будет использовать *One-hot-encoding*, но столбцы *IR2* и *IR3* будут сильно разреженными.\
Сейчас просто закодирую с помощью *Target-encoding*:

In [228]:
df['LotShape_Target'] = 0

lotshape_target = df.groupby('LotShape')['SalePrice'].mean()

for lotshape, target_mean in lotshape_target.items():
    df.loc[df['LotShape'] == lotshape, 'LotShape_Target'] = round(target_mean)

df = df.drop(columns='LotShape')

## LandContour fe
Показывает рельеф земельного участка.

In [229]:
df['LandContour_Rating'] = 0.0

df.loc[df['LandContour'] == 'Bnk', 'LandContour_Rating'] = 1
df.loc[df['LandContour'] == 'Lvl', 'LandContour_Rating'] = 3
df.loc[df['LandContour'] == 'Low', 'LandContour_Rating'] = 4.3
df.loc[df['LandContour'] == 'HLS', 'LandContour_Rating'] = 5.8

df = df.drop(columns='LandContour')

## Utilities fe
Показывает виды доступных коммунальных услуг. Все сэмплы имеют значение *AllPub*, лишь один значение *NoSeWa* — отсутствие воды и канализации.

Признак является квазиконстантным. Следует удалить:

In [230]:
df = df.drop(columns='Utilities')

## LotConfig fe
Конфигурация участка. В принципе, сэмплов каждого значения достаточно для обучения, но значение *FR3* содержит всего-лишь 4 сэмпла. Очень мало для выделения в одну категорию.
Но удалять эти 4 сэмпла тоже не лучший вариант. Лучше объединить их с *FR2* в одну категорию:

In [231]:
df['LotConfig'] = df['LotConfig'].replace(['FR2', 'FR3'], 'FR')

df['LotConfig_Rating'] = 0.0

df.loc[df['LotConfig'] == 'Inside', 'LotConfig_Rating'] = 1
df.loc[df['LotConfig'] == 'FR', 'LotConfig_Rating'] = 1.2
df.loc[df['LotConfig'] == 'Corner', 'LotConfig_Rating'] = 1.3
df.loc[df['LotConfig'] == 'CulDSac', 'LotConfig_Rating'] = 3.6

df = df.drop(columns='LotConfig')

## LandSlope fe
Наклон участка. Значение *Sev* содержит всего-лишь 13 значений. Мало для обучения.

Посмотрим на среднюю цену дома в зависимости от наклона участка:

In [232]:
df.groupby('LandSlope')['SalePrice'].mean()

LandSlope
Gtl    179956.799566
Mod    196734.138462
Sev    204379.230769
Name: SalePrice, dtype: float64

Значения *Mod* и *Sev* оба обозначают наклон участка. Лучше объединить их в одну группу:

In [233]:
df['LandSlope'] = df['LandSlope'].replace(['Sev', 'Mod'], 'Sloped')

df['LandSlope_Rating'] = 0.0

df.loc[df['LandSlope'] == 'Gtl', 'LandSlope_Rating'] = 1
df.loc[df['LandSlope'] == 'Sloped', 'LandSlope_Rating'] = 3.8

df = df.drop(columns='LandSlope')

## Neighborhood fe
Район расположения участка. В EDA уже разделил его на классы, которые показывают отличную корреляцию.\
Только там были указаны классы *Highest*, *High*, *Medium*, *Low* и *Lowest*. Надо их передалать в числовые признаки:

In [234]:
highest_neighborhood = ['NoRidge', 'NridgHt', 'StoneBr']
high_neighborhood = ['Timber', 'Veenker', 'Somerst', 'ClearCr', 'Crawfor']
medium_neighborhood = ['CollgCr', 'Blmngtn', 'NWAmes', 'SawyerW', 'Gilbert', 'Mitchel', 'NPkVill', 'NAmes']
low_neighborhood = ['Sawyer', 'SWISU', 'Blueste', 'BrkSide', 'Edwards', 'OldTown', 'BrDale']
lowest_neighborhood = ['IDOTRR', 'MeadowV']

df['Neighborhood_Class'] = 5.9 # Среднее значение по умолчанию

df.loc[df['Neighborhood'].isin(lowest_neighborhood), 'Neighborhood_Class'] = 1
df.loc[df['Neighborhood'].isin(low_neighborhood), 'Neighborhood_Class'] = 2.5
df.loc[df['Neighborhood'].isin(medium_neighborhood), 'Neighborhood_Class'] = 4.7
df.loc[df['Neighborhood'].isin(high_neighborhood), 'Neighborhood_Class'] = 7.2
df.loc[df['Neighborhood'].isin(highest_neighborhood), 'Neighborhood_Class'] = 11.8

df = df.drop(columns='Neighborhood')

## Condition1 fe
Близость к объектам инфраструктуры.

Разбиение уже сделал в EDA, осталось только перевести в числа:

In [235]:
good_condition1 = ['PosN', 'PosA', 'RRNn', 'RRNe']
normal_condition1 = ['Norm', 'RRAn']
poor_condition1 = ['Feedr', 'Artery', 'RRAe']

df['Condition1_Rating'] = 0.0

df.loc[df['Condition1'].isin(poor_condition1), 'Condition1_Rating'] = 1
df.loc[df['Condition1'].isin(normal_condition1), 'Condition1_Rating'] = 4.5
df.loc[df['Condition1'].isin(good_condition1), 'Condition1_Rating'] = 6.8

df = df.drop(columns='Condition1')

## Condition2 fe
Аналогично *Condition1*:

In [236]:
good_condition2 = ['PosN', 'PosA', 'RRNn', 'RRNe']
normal_condition2 = ['Norm', 'RRAn']
poor_condition2 = ['Feedr', 'Artery', 'RRAe']

df['Condition2_Rating'] = 0.0

df.loc[df['Condition2'].isin(poor_condition2), 'Condition2_Rating'] = 1
df.loc[df['Condition2'].isin(normal_condition2), 'Condition2_Rating'] = 2.6
df.loc[df['Condition2'].isin(good_condition2), 'Condition2_Rating'] = 3.7

df = df.drop(columns='Condition2')

## BldgType fe
Тип жилища.

In [237]:
df['BldgType_Rating'] = 0.0

df.loc[df['BldgType'] == '2fmCon', 'BldgType_Rating'] = 1
df.loc[df['BldgType'] == 'Duplex', 'BldgType_Rating'] = 1.3
df.loc[df['BldgType'] == 'Twnhs', 'BldgType_Rating'] = 1.4
df.loc[df['BldgType'] == 'TwnhsE', 'BldgType_Rating'] = 4.3
df.loc[df['BldgType'] == '1Fam', 'BldgType_Rating'] = 4.5

df = df.drop(columns='BldgType')

## HouseStyle fe
Архитектурная конфигурация дома. 8 значений. Достаточно много, хочется их уменьшить. В данном случае есть такая возможность:
- *2.5Unf* и *2.5Fin* можно объединить в 2.5-этажные дома *2.5Story*;
- *1.5Unf* и *1.5Fin* можно объединить в 1.5-этажные дома *1.5Story*;
- Разделённый дома *SFoyer* и *SLvl* можно объединить в просто распиленный дома *Split*.

In [238]:
df['HouseStyle'] = df['HouseStyle'].replace(['2.5Unf', '2.5Fin'], '2.5Story')
df['HouseStyle'] = df['HouseStyle'].replace(['1.5Unf', '1.5Fin'], '1.5Story')
df['HouseStyle'] = df['HouseStyle'].replace(['SFoyer', 'SLvl'], 'Split')

df['HouseStyle_Rating'] = 0.0

df.loc[df['HouseStyle'] == '1.5Story', 'HouseStyle_Rating'] = 1
df.loc[df['HouseStyle'] == 'Split', 'HouseStyle_Rating'] = 2.2
df.loc[df['HouseStyle'] == '1Story', 'HouseStyle_Rating'] = 3.7
df.loc[df['HouseStyle'] == '2.5Story', 'HouseStyle_Rating'] = 4.6
df.loc[df['HouseStyle'] == '2Story', 'HouseStyle_Rating'] = 6.6

df = df.drop(columns='HouseStyle')

## RoofStyle fe
Тип крыши.

Самые поплярные типы крыш: *Gable*, *Hip*. Остальных очень мало, стоит объединить в одну группу:

In [239]:
df['RoofStyle'] = df['RoofStyle'].replace(['Gambrel', 'Mansard', 'Flat', 'Shed'], 'Rare')

df['RoofStyle_Rating'] = 0.0

df.loc[df['RoofStyle'] == 'Gable', 'RoofStyle_Rating'] = 1
df.loc[df['RoofStyle'] == 'Rare', 'RoofStyle_Rating'] = 1.3
df.loc[df['RoofStyle'] == 'Hip', 'RoofStyle_Rating'] = 3.4

df = df.drop(columns='RoofStyle')

## RoofMatl fe
Материал крыши.

Самый поплярный — битумная черепица *CompShg*. Остальных очень мало, стоит объединить в одну группу:

In [240]:
df['RoofMatl'] = df['RoofMatl'].replace(['ClyTile', 'Membran', 'Metal', 'Roll', 'Tar&Grv', 'WdShake', 'WdShngl'], 'Rare')

df['RoofMatl_Rating'] = 0.0

df.loc[df['RoofMatl'] == 'CompShg', 'RoofMatl_Rating'] = 1
df.loc[df['RoofMatl'] == 'Rare', 'RoofMatl_Rating'] = 7.2

df = df.drop(columns='RoofMatl')

## Exterior1st fe
Внешняя отделка дома.

Много всяких значений и много значений, у которых мало сэмплов. В EDA уже объединил в группы:

In [241]:
df['Exterior1st_Rating'] = 0.0

outdated_exterior1st = ['AsbShng', 'BrkComm', 'AsphShn', 'CBlock', 'BrkComm', 'PreCast', 'Other']
wooden_exterior1st = ['Wd Sdng', 'Plywood', 'WdShing']
standart_exterior1st = ['VinylSd', 'MetalSd', 'HdBoard']
premium_exterior1st = ['BrkFace', 'Stone', 'CemntBd', 'Stucco', 'ImStucc']

df.loc[df['Exterior1st'].isin(outdated_exterior1st), 'Exterior1st_Rating'] = 1
df.loc[df['Exterior1st'].isin(wooden_exterior1st), 'Exterior1st_Rating'] = 3.5
df.loc[df['Exterior1st'].isin(standart_exterior1st), 'Exterior1st_Rating'] = 4
df.loc[df['Exterior1st'].isin(premium_exterior1st), 'Exterior1st_Rating'] = 5

df = df.drop(columns='Exterior1st')

## Exterior2nd fe
Аналогично *Exterior1st*:

In [242]:
df['Exterior2nd_Rating'] = 0.0

outdated_exterior2st = ['AsbShng', 'BrkComm', 'AsphShn', 'CBlock', 'BrkComm', 'PreCast', 'Other']
wooden_exterior2st = ['Wd Sdng', 'Plywood', 'WdShing']
standart_exterior2st = ['VinylSd', 'MetalSd', 'HdBoard']
premium_exterior2st = ['BrkFace', 'Stone', 'CemntBd', 'Stucco', 'ImStucc']

df.loc[df['Exterior2nd'].isin(outdated_exterior2st), 'Exterior2nd_Rating'] = 1
df.loc[df['Exterior2nd'].isin(wooden_exterior2st), 'Exterior2nd_Rating'] = 3.5
df.loc[df['Exterior2nd'].isin(standart_exterior2st), 'Exterior2nd_Rating'] = 4
df.loc[df['Exterior2nd'].isin(premium_exterior2st), 'Exterior2nd_Rating'] = 5

df = df.drop(columns='Exterior2nd')

## MasVnrType fe
Декоративная облицовка дома.

Значение *CBlock* отсутствует в тренировочных сэмплах, установлю для него значение по умолчанию:

In [243]:
df['MasVnrType_Rating'] = 2.75 # Значение по умолчанию

df.loc[df['MasVnrType'] == 'BrkCmn', 'MasVnrType_Rating'] = 1
df.loc[df['MasVnrType'] == 'Absent', 'MasVnrType_Rating'] = 1.4
df.loc[df['MasVnrType'] == 'BrkFace', 'MasVnrType_Rating'] = 3.2
df.loc[df['MasVnrType'] == 'Stne', 'MasVnrType_Rating'] = 5.5

df = df.drop(columns='MasVnrType')

## ExterQual fe
Качество материала внешней отделки.

Сэмплы со значением *Poor* вообще отсутствуют, а *Fair* всего-лишь 14 штук, что составляет <1% от данных. При этом, *Poor* и *Fair* являются смежными признаками.
Следует их объединить:

In [244]:
df['ExterQual_Rating'] = 1.0 # Сразу объединяю и присваиваю значение для Poor & Fair

df.loc[df['ExterQual'] == 'TA', 'ExterQual_Rating'] = 2.7
df.loc[df['ExterQual'] == 'Gd', 'ExterQual_Rating'] = 5.2
df.loc[df['ExterQual'] == 'Ex', 'ExterQual_Rating'] = 9

df = df.drop(columns='ExterQual')

## ExterCond fe
Оценивает текущее состояние внешнего материала внешней отделки.

Очень мало сэмплов для значений *Excellent* и *Poor*. Стоит объединить *Poor* и *Fair*, а также *Good* и *Excellent*:

In [245]:
low_extercond = ['Po', 'Fa']
medium_extercond = ['TA']
high_extercond = ['Gd', 'Ex']

df['ExterCond_Rating'] = 0.0

df.loc[df['ExterCond'].isin(low_extercond), 'ExterCond_Rating'] = 1
df.loc[df['ExterCond'].isin(medium_extercond), 'ExterCond_Rating'] = 3
df.loc[df['ExterCond'].isin(high_extercond), 'ExterCond_Rating'] = 3.5

df = df.drop(columns='ExterCond')

## Foundation fe
Тип фундамента.

Значений *Wood*, *Stone* и *Slab* очень мало. Объединю их в один:

In [246]:
rare_foundation = ['Wood', 'Stone', 'Slab']
df.loc[df['Foundation'].isin(rare_foundation), 'Foundation'] = 'Rare'

df['Foundation_Rating'] = 0.0

df.loc[df['Foundation'] == 'Rare', 'Foundation_Rating'] = 1
df.loc[df['Foundation'] == 'BrkTil', 'Foundation_Rating'] = 1.3
df.loc[df['Foundation'] == 'CBlock', 'Foundation_Rating'] = 3.1
df.loc[df['Foundation'] == 'PConc', 'Foundation_Rating'] = 6.3

df = df.drop(columns='Foundation')

## BsmtQual fe
Высота подвала.

Имеет 6 значений, в тренировочных данных отсутствует значение *Poor*. Данное значение близко к *Fair*, поэтому если в новых данных попадётся *Poor*, назначу ему значение *Fair*:

In [249]:
df['BsmtQual_Ratin'] = 1.5 # Значение по умолчанию для Poor

df.loc[df['BsmtQual'] == 'Absent', 'BsmtQual_Rating'] = 1
df.loc[df['BsmtQual'] == 'Fa', 'BsmtQual_Rating'] = 1.5
df.loc[df['BsmtQual'] == 'TA', 'BsmtQual_Rating'] = 3.6
df.loc[df['BsmtQual'] == 'Gd', 'BsmtQual_Rating'] = 5.6
df.loc[df['BsmtQual'] == 'Ex', 'BsmtQual_Rating'] = 12.4

df = df.drop(columns='BsmtQual')

## BsmtCond fe